# Phase 2 (수정본) — 평가 버그 픽스

원래 결과에서 **FP16(베이스)가 양자화 모델보다 낮게** 나오는 비정상 현상이 있었음. 원인은 평가 코드 버그.

**고친 점 3가지**
1. **채팅 템플릿 적용** — 평가 시 `messages[0]['content']`(raw)를 넣던 것을 학습과 동일하게 `apply_chat_template(..., add_generation_prompt=True)`로 변경. **(가장 핵심)**
2. **greedy 디코딩** — `num_beams=2`, `length_penalty=1.2` 제거 → ROUGE-L 길이 왜곡 방지 (Phase 1과 일치).
3. **전체 재측정** — `FORCE_RERUN=True`로 옫 캐시 무시하고 7개 비트 모두 새로 측정. 측정은 저비트(역순)부터.

> 실행 전: 드라이브에 `phase1_winner.json`과 우승 어댑터가 있어야 합니다(셀 1~2가 읽음).


In [ ]:
!pip install -q --upgrade transformers torchao rouge-score bitsandbytes accelerate peft hqq
!pip install -q kiwipiepy bert-score   # ← 추가 (kiwipiepy: Mecab보다 Colab 안정적)
import torch
print(f"✓ Torch: {torch.__version__}")
print(f"✓ TorchAO 업그레이드 완료")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, gc, time
import torch, psutil
from tqdm import tqdm
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_DIR    = '/content/drive/MyDrive/RuralVet-LLM'
DATA_DIR    = f'{BASE_DIR}/data'
RESULTS_DIR = f'{BASE_DIR}/results'
WORK_DIR    = '/content/work'
PHASE2_DIR  = f'{RESULTS_DIR}/phase2'
EVAL_DIR    = f'{PHASE2_DIR}/eval_per_bit'

for d in (WORK_DIR, PHASE2_DIR, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

QUANT_TYPES = [
    'FP16_Baseline',
    'Float8_TorchAO',
    '8bit_Standard',
    '8bit_TorchAO',
    '4bit_NF4',
    '4bit_Pure_Float',
    '2bit_HQQ',
]

with open(f'{RESULTS_DIR}/phase1_winner.json', encoding='utf-8') as f:
    info = json.load(f)
WINNER       = info['phase1_winner']
ADAPTER_PATH = info.get('adapter_path', f'{RESULTS_DIR}/phase1/{WINNER}/adapter')
MERGED_DIR   = f'{WORK_DIR}/{WINNER}_merged_fp16'
print(f"✓ Phase 1 우승: {WINNER}")

In [ ]:
if os.path.exists(MERGED_DIR):
    print(f"병합본 이미 존재 → 스킵")
else:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(
        'Qwen/Qwen3.5-4B', torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True
    )
    tok = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-4B', trust_remote_code=True)
    merged = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()
    os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR)
    tok.save_pretrained(MERGED_DIR)
    print(f"✓ FP16 완본 저장 완료: {MERGED_DIR}")
    del base, merged, tok
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

eval_raw   = load_jsonl(f'{DATA_DIR}/eval.jsonl')
references = [next(m['content'] for m in s['messages'] if m['role'] == 'assistant')
              for s in eval_raw]

# ── 1순위: 한국어 형태소 기반 ROUGE ─────────────────────────────────────────
from kiwipiepy import Kiwi
_kiwi    = Kiwi()
_rscorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def tokenize_ko(text):
    return ' '.join(t.form for t in _kiwi.tokenize(text))

def calculate_rouge_l(preds):
    scores = [
        _rscorer.score(tokenize_ko(ref), tokenize_ko(p))['rougeL'].fmeasure
        for p, ref in zip(preds, references)
    ]
    return sum(scores) / len(scores)

# ── 3순위: BERTScore ─────────────────────────────────────────────────────────
def calculate_bert_score(preds):
    from bert_score import score as _bert_score
    _, _, F1 = _bert_score(preds, references,
                           lang='ko',        # model_type 제거, lang만 사용
                           verbose=False)
    return round(F1.mean().item(), 4)

# ── 메인 평가 함수 ────────────────────────────────────────────────────────────
NUM_BEAMS = 2  # 2순위: beam search (greedy=1, 빠름↔품질 트레이드오프)

def run_native_quant_eval(quant_type):
    tok = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)

    bnb_config = None
    if quant_type == '8bit_Standard':
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    elif quant_type == '4bit_NF4':
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                        bnb_4bit_compute_dtype=torch.float16)
    elif quant_type == '4bit_Pure_Float':
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="fp4",
                                        bnb_4bit_compute_dtype=torch.float16)

    start_ram = psutil.virtual_memory().used / 1e9
    torch.cuda.empty_cache()
    vram_before = torch.cuda.memory_allocated()

    if quant_type == '2bit_HQQ':
        from hqq.models.hf.base import AutoHQQHFModel
        from hqq.core.quantize import BaseQuantizeConfig
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        AutoHQQHFModel.quantize_model(model, quant_config=BaseQuantizeConfig(nbits=2, group_size=64))

    elif quant_type == 'Float8_TorchAO':
        from torchao.quantization import quantize_, Float8WeightOnlyConfig
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        quantize_(model, Float8WeightOnlyConfig())

    elif quant_type == '8bit_TorchAO':
        from torchao.quantization import quantize_, Int8WeightOnlyConfig
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        quantize_(model, Int8WeightOnlyConfig())

    else:
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, quantization_config=bnb_config,
            torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)

    model_size_gb = (torch.cuda.memory_allocated() - vram_before) / 1e9

    preds, total_tokens = [], 0
    start_time = time.time()
    peak_ram   = start_ram

    for sample in tqdm(eval_raw, desc=f"[{quant_type}]"):
        # [FIX 1] 학습 때와 동일하게 채팅 템플릿 적용 (raw content 사용 금지)
        msgs   = [m for m in sample['messages'] if m['role'] != 'assistant']
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to("cuda")
        cur_ram = psutil.virtual_memory().used / 1e9
        if cur_ram > peak_ram: peak_ram = cur_ram

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=256,
                # [FIX 2] greedy 디코딩 (beam/length_penalty 제거 → ROUGE-L 길이 왜곡 방지, Phase 1과 일치)
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )

        gen = out[0][inputs.input_ids.shape[1]:]
        total_tokens += len(gen)
        preds.append(tok.decode(gen, skip_special_tokens=True))

    elapsed     = time.time() - start_time
    tok_per_sec = total_tokens / elapsed if elapsed > 0 else 0

    print(f"  → ROUGE-L 계산 중...")
    rouge = calculate_rouge_l(preds)
    print(f"  → BERTScore 계산 중...")
    bscore = calculate_bert_score(preds)

    save_path = f"{PHASE2_DIR}/models/{quant_type}"
    os.makedirs(save_path, exist_ok=True)
    model.save_pretrained(save_path)
    tok.save_pretrained(save_path)

    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "rouge_l":       round(rouge, 4),
        "bert_score":    bscore,
        "model_size_gb": round(model_size_gb, 2),
        "peak_ram_gb":   round(peak_ram - start_ram, 2),
        "cpu_tok_s":     round(tok_per_sec, 2),
    }

In [ ]:
MODELS_EVAL_DATA = {}

# [FIX 3] 이전 버그 결과 캐시가 남아 있을 수 있으므로 전부 새로 측정.
#         (중간에 끊겨 이어서 돌릴 땐 False 로 바꾸면 완료된 비트는 건너뜀)
FORCE_RERUN = True

# 노형우님 요청: 저비트부터(역순) 측정 → 핵심 결과부터 확보
for q in reversed(QUANT_TYPES):
    path = f"{EVAL_DIR}/{q}.json"
    if os.path.exists(path) and not FORCE_RERUN:
        with open(path) as f:
            MODELS_EVAL_DATA[q] = json.load(f)
        print(f"[{q}] 기존 결과 로드")
        continue

    print(f"\n▶ [{q}] 측정 시작...")
    try:
        m = run_native_quant_eval(q)
        m["label"] = q
        with open(path, "w", encoding="utf-8") as f:
            json.dump(m, f, indent=2)
        MODELS_EVAL_DATA[q] = m
        print(f"  ✓ {m}")
    except Exception as e:
        print(f"  ❌ 실패: {e}")



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)

import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
fm._load_fontmanager(try_read_cache=False)

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 데이터 정리 ──────────────────────────────────────────────────────────────
df = pd.DataFrame([MODELS_EVAL_DATA[q] for q in QUANT_TYPES if q in MODELS_EVAL_DATA]).set_index('label')

fp16_rouge = df.loc['FP16_Baseline', 'rouge_l']
size_col = 'model_size_gb' if df['model_size_gb'].max() > 0 else 'peak_ram_gb'

df['quality_retention'] = (df['rouge_l'] / fp16_rouge * 100).round(1)
df['efficiency_score']  = (
    (df['rouge_l'] / fp16_rouge) * df['cpu_tok_s'] /
    df[size_col].replace(0, 0.01)
).round(3)

print("=" * 100)
print("Phase 2 양자화 종합 결과")
print("=" * 100)
fp16_bert = df.loc['FP16_Baseline', 'bert_score']
df['bert_retention'] = (df['bert_score'] / fp16_bert * 100).round(1)

display(df[['rouge_l', 'bert_score', 'quality_retention', 'bert_retention',
            'model_size_gb', 'peak_ram_gb', 'cpu_tok_s', 'efficiency_score']])

# ── 시나리오별 우승자 ────────────────────────────────────────────────────────
def best_or_none(subset, col, maximize=True):
    if subset.empty: return None
    return subset[col].idxmax() if maximize else subset[col].idxmin()

scenarios = {
    "저사양 엣지  (RAM ≤ 4GB)":          best_or_none(df[df['peak_ram_gb'] <= 4.0],  'quality_retention'),
    "균형형      (RAM ≤ 6GB, ≥10tok/s)":  best_or_none(df[(df['peak_ram_gb'] <= 6.0) & (df['cpu_tok_s'] >= 10.0)], 'efficiency_score'),
    "고품질 서버  (품질 보존 ≥ 90%)":     best_or_none(df[df['quality_retention'] >= 90.0], size_col, maximize=False),
    "최소 크기    (크기 무조건 최소)":     best_or_none(df, size_col, maximize=False),
}

print("\n" + "=" * 100)
print("시나리오별 최적 양자화")
print("=" * 100)
for scenario, winner in scenarios.items():
    if winner:
        r = df.loc[winner]
        print(f"  {scenario:38s} → {winner:20s}"
              f"  ROUGE-L={r['rouge_l']:.4f}"
              f"  품질보존={r['quality_retention']:.1f}%"
              f"  RAM={r['peak_ram_gb']:.1f}GB"
              f"  {r['cpu_tok_s']:.1f}tok/s")
    else:
        print(f"  {scenario:38s} → 조건 충족 모델 없음")

# ── Pareto 프론티어 ──────────────────────────────────────────────────────────
def pareto_optimal(df):
    result = []
    for i, (idx, r) in enumerate(df.iterrows()):
        dominated = any(
            r2['rouge_l'] >= r['rouge_l'] and
            r2['cpu_tok_s'] >= r['cpu_tok_s'] and
            r2[size_col] <= r[size_col] and
            r2['peak_ram_gb'] <= r['peak_ram_gb'] and
            (r2['rouge_l'] > r['rouge_l'] or r2['cpu_tok_s'] > r['cpu_tok_s'] or
             r2[size_col] < r[size_col] or r2['peak_ram_gb'] < r['peak_ram_gb'])
            for j, (_, r2) in enumerate(df.iterrows()) if i != j
        )
        if not dominated:
            result.append(idx)
    return result

pareto_models = pareto_optimal(df)
print(f"\n▶ Pareto 최적 모델: {pareto_models}")

# ── 시각화 ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

clr = ['#2ecc71' if idx in pareto_models else '#bdc3c7' for idx in df.index]
xtick_kw = dict(rotation=45, ha='right', fontsize=8)

# 1. ROUGE-L
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(df.index, df['rouge_l'], color=clr, alpha=0.9)
ax1.axhline(fp16_rouge, color='red', ls='--', lw=1.2, label='FP16 기준')
ax1.set_title('ROUGE-L (품질)', fontweight='bold')
ax1.set_xticklabels(df.index, **xtick_kw)
ax1.legend(fontsize=8)

# 2. 품질 보존율
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(df.index, df['quality_retention'], color=clr, alpha=0.9)
ax2.axhline(90,  color='orange', ls='--', lw=1.2, label='90% 선')
ax2.axhline(100, color='red',    ls='--', lw=1.2, label='FP16 (100%)')
ax2.set_title('품질 보존율 (%)', fontweight='bold')
ax2.set_xticklabels(df.index, **xtick_kw)
ax2.legend(fontsize=8)

# 3. 모델 크기 (GPU VRAM)
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(df.index, df[size_col], color=clr, alpha=0.9)
ax3.set_title(f'모델 크기 ({size_col}, GB)', fontweight='bold')
ax3.set_xticklabels(df.index, **xtick_kw)

# 4. 추론 속도
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(df.index, df['cpu_tok_s'], marker='o', color='crimson', lw=2)
ax4.axhline(10, color='gray', ls='--', lw=1, label='10 tok/s')
ax4.set_title('추론 속도 (tok/s)', fontweight='bold')
ax4.set_xticklabels(df.index, **xtick_kw)
ax4.legend(fontsize=8)

# 5. 효율 점수
ax5 = fig.add_subplot(gs[1, 1])
ax5.bar(df.index, df['efficiency_score'], color=clr, alpha=0.9)
ax5.set_title('효율 점수 (품질×속도/크기)', fontweight='bold')
ax5.set_xticklabels(df.index, **xtick_kw)

# 6. Pareto 산점도 (품질 vs 크기)
ax6 = fig.add_subplot(gs[1, 2])
for idx, row in df.iterrows():
    c = '#2ecc71' if idx in pareto_models else '#bdc3c7'
    ax6.scatter(row[size_col], row['rouge_l'], color=c, s=90, zorder=5)
    ax6.annotate(idx, (row[size_col], row['rouge_l']),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)

# Pareto 경계선
pdf = df.loc[pareto_models].sort_values(size_col)
ax6.plot(pdf[size_col], pdf['rouge_l'], 'g--', lw=1.2, alpha=0.6)
ax6.set_xlabel('모델 크기 (GB)')
ax6.set_ylabel('ROUGE-L')
ax6.set_title('Pareto 프론티어', fontweight='bold')

from matplotlib.patches import Patch
ax6.legend(handles=[Patch(facecolor='#2ecc71', label='Pareto 최적'),
                    Patch(facecolor='#bdc3c7', label='비최적')], fontsize=8)

plt.suptitle('Phase 2 양자화 종합 분석', fontsize=15, fontweight='bold')
plt.savefig(f'{PHASE2_DIR}/phase2_result.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 결과 저장 ────────────────────────────────────────────────────────────────
phase2_result = {
    "phase1_winner":       WINNER,
    "fp16_rouge_baseline": float(fp16_rouge),
    "scenarios":           {k: v for k, v in scenarios.items() if v},
    "pareto_optimal":      pareto_models,
    "all_metrics":         df.reset_index().to_dict('records'),
}
with open(f'{RESULTS_DIR}/phase2_winner.json', 'w', encoding='utf-8') as f:
    json.dump(phase2_result, f, indent=2, ensure_ascii=False,
              default=lambda x: float(x) if hasattr(x, 'item') else str(x))

print(f"\n✓ phase2_winner.json 저장 완료. Phase 3으로 이동 가능.")

In [ ]:
# ── Cell 6: 심층 시각화 (7종 차트) ───────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from math import pi

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.DataFrame([MODELS_EVAL_DATA[q] for q in QUANT_TYPES if q in MODELS_EVAL_DATA]).set_index('label')

# ← 이 두 줄 추가
size_col = 'model_size_gb' if df['model_size_gb'].max() > 0 else 'peak_ram_gb'
df['efficiency_score'] = ((df['rouge_l'] / fp16_rouge) * df['cpu_tok_s'] / df[size_col].replace(0, 0.01)).round(3)

fp16_rouge = df.loc['FP16_Baseline', 'rouge_l']
fp16_bert  = df.loc['FP16_Baseline', 'bert_score']
fp16_speed = df.loc['FP16_Baseline', 'cpu_tok_s']
fp16_size  = df.loc['FP16_Baseline', 'model_size_gb']

COLORS = {
    'FP16_Baseline':  '#95a5a6',
    'Float8_TorchAO': '#3498db',
    '8bit_Standard':  '#9b59b6',
    '8bit_TorchAO':   '#8e44ad',
    '4bit_NF4':       '#2ecc71',
    '4bit_Pure_Float':'#27ae60',
    '2bit_HQQ':       '#e74c3c',
}

fig = plt.figure(figsize=(22, 18))
fig.suptitle('Phase 2 양자화 심층 분석', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.4)

# ── 1. 레이더 차트 ─────────────────────────────────────────────────────────────
ax_r = fig.add_subplot(gs[0, 0], polar=True)
categories = ['BERTScore\n보존율', '추론속도\n정규화', '크기효율\n(1/크기)', 'RAM효율\n(1/RAM)', 'ROUGE-L\n정규화']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

for idx, row in df.iterrows():
    bert_ret  = row['bert_score'] / fp16_bert
    speed_ret = min(row['cpu_tok_s'] / fp16_speed, 2.0)
    size_eff  = min(fp16_size / max(row['model_size_gb'], 0.1), 2.0)
    ram_max   = df['peak_ram_gb'].max()
    ram_eff   = min((ram_max - row['peak_ram_gb']) / max(ram_max, 0.1) + 0.5, 2.0)
    rouge_ret = min(row['rouge_l'] / fp16_rouge, 2.0)
    vals = [bert_ret, speed_ret, size_eff, ram_eff, rouge_ret] + [bert_ret]
    ax_r.plot(angles, vals, linewidth=1.8, label=idx, color=COLORS.get(idx, '#999'))
    ax_r.fill(angles, vals, alpha=0.08, color=COLORS.get(idx, '#999'))

ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(categories, fontsize=7.5)
ax_r.set_ylim(0, 2.2)
ax_r.axhline(1.0, color='red', lw=0.8, ls='--', alpha=0.5)
ax_r.set_title('다차원 성능 레이더\n(붉은 점선 = FP16 기준)', fontweight='bold', pad=15, fontsize=9)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.55, 1.25), fontsize=6.5)

# ── 2. 버블 차트: 크기 × BERTScore × 속도 ─────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
for idx, row in df.iterrows():
    ax_b.scatter(row['model_size_gb'], row['bert_score'],
                 s=row['cpu_tok_s'] * 9, color=COLORS.get(idx, '#bdc3c7'),
                 alpha=0.78, zorder=5, edgecolors='white', lw=1)
    ax_b.annotate(idx.replace('_', '\n'), (row['model_size_gb'], row['bert_score']),
                  textcoords='offset points', xytext=(6, 3), fontsize=6.5)
ax_b.axhline(fp16_bert, color='red', ls='--', lw=1, alpha=0.6, label=f'FP16 ({fp16_bert:.4f})')
for spd, lbl in [(15, '15 tok/s'), (35, '35 tok/s'), (55, '55 tok/s')]:
    ax_b.scatter([], [], s=spd * 9, c='#aaa', alpha=0.55, label=lbl)
ax_b.set_xlabel('모델 크기 (GB)', fontsize=9)
ax_b.set_ylabel('BERTScore', fontsize=9)
ax_b.set_title('버블 차트\n크기(x) × 품질(y) × 속도(원 크기)', fontweight='bold', fontsize=9)
ax_b.legend(fontsize=6.5, loc='lower right')

# ── 3. ROUGE-L vs BERTScore 산점도 ────────────────────────────────────────────
ax_s = fig.add_subplot(gs[0, 2])
for idx, row in df.iterrows():
    ax_s.scatter(row['rouge_l'], row['bert_score'],
                 color=COLORS.get(idx, '#bdc3c7'), s=120, zorder=5, edgecolors='white', lw=1)
    ax_s.annotate(idx.replace('_', '\n'), (row['rouge_l'], row['bert_score']),
                  textcoords='offset points', xytext=(4, 3), fontsize=6.5)
ax_s.axvline(fp16_rouge, color='red',  ls='--', lw=1, alpha=0.6, label=f'FP16 ROUGE-L ({fp16_rouge:.4f})')
ax_s.axhline(fp16_bert,  color='blue', ls='--', lw=1, alpha=0.6, label=f'FP16 BERTScore ({fp16_bert:.4f})')
ax_s.set_xlabel('ROUGE-L', fontsize=9)
ax_s.set_ylabel('BERTScore', fontsize=9)
ax_s.set_title('ROUGE-L vs BERTScore\n두 지표의 불일치 시각화', fontweight='bold', fontsize=9)
ax_s.legend(fontsize=7)

# ── 4. 정규화 히트맵 ──────────────────────────────────────────────────────────
ax_h = fig.add_subplot(gs[1, :2])
metrics_dict = {
    'BERTScore':              df['bert_score'] / df['bert_score'].max(),
    'ROUGE-L':                df['rouge_l']    / df['rouge_l'].max(),
    '추론속도 (tok/s)':       df['cpu_tok_s']  / df['cpu_tok_s'].max(),
    '크기 효율 (↓작을수록)':  (1 / df['model_size_gb']) / (1 / df['model_size_gb']).max(),
    'RAM 효율 (↓적을수록)':   (1 / df['peak_ram_gb'].replace(0, 0.01)) / (1 / df['peak_ram_gb'].replace(0, 0.01)).max(),
    '종합 효율 점수':          df['efficiency_score'] / df['efficiency_score'].max(),
}
hm_df = pd.DataFrame(metrics_dict).T
im = ax_h.imshow(hm_df.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax_h.set_xticks(range(len(df.index)))
ax_h.set_xticklabels(df.index, rotation=28, ha='right', fontsize=8.5)
ax_h.set_yticks(range(len(metrics_dict)))
ax_h.set_yticklabels(list(metrics_dict.keys()), fontsize=8.5)
for i in range(len(metrics_dict)):
    for j in range(len(df.index)):
        v = hm_df.values[i, j]
        ax_h.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                  color='black' if 0.25 < v < 0.75 else 'white')
plt.colorbar(im, ax=ax_h, fraction=0.015, pad=0.02)
ax_h.set_title('지표별 정규화 히트맵  (1.00 = 각 지표 최고값)', fontweight='bold', fontsize=10)

# ── 5. FP16 대비 델타 차트 ────────────────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 2])
non_fp16    = df.drop('FP16_Baseline')
delta_bert  = ((non_fp16['bert_score'] - fp16_bert)  / fp16_bert  * 100).round(1)
delta_rouge = ((non_fp16['rouge_l']    - fp16_rouge) / fp16_rouge * 100).round(1)
x = np.arange(len(non_fp16))
w = 0.35
ax_d.bar(x - w/2, delta_bert,
         w, label='BERTScore Δ%',
         color=['#2ecc71' if v >= 0 else '#e74c3c' for v in delta_bert], alpha=0.85)
ax_d.bar(x + w/2, delta_rouge,
         w, label='ROUGE-L Δ%',
         color=['#3498db' if v >= 0 else '#e67e22' for v in delta_rouge], alpha=0.85)
ax_d.axhline(0, color='black', lw=0.8)
ax_d.set_xticks(x)
ax_d.set_xticklabels(non_fp16.index, rotation=38, ha='right', fontsize=7)
ax_d.set_ylabel('FP16 대비 변화율 (%)', fontsize=8)
ax_d.set_title('FP16 대비 품질 변화\n(+ = FP16보다 높음)', fontweight='bold', fontsize=9)
ax_d.legend(fontsize=7)
for i, v_b in enumerate(delta_bert):
    offset = 1.5 if v_b >= 0 else -4.5
    ax_d.text(i - w/2, v_b + offset, f'{v_b:+.1f}', ha='center', fontsize=6.5)

# ── 6. 크기 감소 vs BERTScore 보존 트레이드오프 ───────────────────────────────
ax_t = fig.add_subplot(gs[2, :2])
size_red   = ((fp16_size - non_fp16['model_size_gb']) / fp16_size * 100)
bert_ret_p = (non_fp16['bert_score'] / fp16_bert * 100)

ax_t.fill_between([30, 75], [90, 90], [106, 106], alpha=0.07, color='#2ecc71', label='이상적 영역')
for idx, (sr, br) in zip(non_fp16.index, zip(size_red, bert_ret_p)):
    ax_t.scatter(sr, br, color=COLORS.get(idx, '#bdc3c7'), s=150, zorder=5, edgecolors='white', lw=1.2)
    ax_t.annotate(idx, (sr, br), textcoords='offset points', xytext=(6, 4), fontsize=8.5)
ax_t.axhline(90,  color='orange', ls='--', lw=1.2, alpha=0.8, label='품질보존 90% 기준선')
ax_t.axhline(100, color='red',    ls='--', lw=1,   alpha=0.5, label='FP16 기준 (100%)')
ax_t.set_xlabel('모델 크기 감소율 (%)', fontsize=10)
ax_t.set_ylabel('BERTScore 보존율 (%)', fontsize=10)
ax_t.set_title('크기 감소 vs 품질 보존 트레이드오프\n(오른쪽 위 = 이상적)', fontweight='bold', fontsize=10)
ax_t.legend(fontsize=8)
ax_t.set_xlim(-5, 80)
ax_t.set_ylim(85, 107)
ax_t.grid(True, alpha=0.2)

# ── 7. 종합 가중치 순위 ────────────────────────────────────────────────────────
ax_rank = fig.add_subplot(gs[2, 2])
ram_inv = 1 / df['peak_ram_gb'].replace(0, 0.01)
composite = (
    df['bert_score'] / fp16_bert                             * 0.45 +
    (df['cpu_tok_s'] / fp16_speed).clip(upper=2)             * 0.25 +
    (fp16_size / df['model_size_gb']).clip(upper=3) / 3      * 0.20 +
    (ram_inv / ram_inv.max())                                 * 0.10
)
composite_sorted = composite.sort_values(ascending=True)
bars = ax_rank.barh(composite_sorted.index, composite_sorted.values,
                    color=[COLORS.get(i, '#bdc3c7') for i in composite_sorted.index],
                    alpha=0.87, edgecolor='white', height=0.55)
for bar, val in zip(bars, composite_sorted.values):
    ax_rank.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                 f'{val:.3f}', va='center', fontsize=8.5)
ax_rank.axvline(composite['FP16_Baseline'], color='red', ls='--', lw=1.2,
                alpha=0.7, label='FP16 기준')
ax_rank.set_xlabel('종합 점수\n(품질45% + 속도25% + 크기20% + RAM10%)', fontsize=7.5)
ax_rank.set_title('종합 가중치 순위', fontweight='bold', fontsize=10)
ax_rank.legend(fontsize=7.5)
ax_rank.grid(axis='x', alpha=0.2)

plt.savefig(f'{PHASE2_DIR}/phase2_advanced_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 심층 시각화 저장 완료:", f'{PHASE2_DIR}/phase2_advanced_viz.png')